# 垃圾分类多源数据集全流程整合管线 (Master Pipeline)
本笔记本将完整的数据预处理流线划分为 5 个独立模块，确保数据无泄漏且高质量融合：
**1. 环境与路径配置**
**2. 全局路径扫描与动态映射**
**3. 跨数据集防泄漏过滤 (pHash Cross-Deduplication)**
**4. 图像自适应裁剪、缩放与最终排重工具**
**5. 核心合并与生成引擎**

> **💡 提示**: 在开始之前，请确保已正确安装 `opencv-python`, `pandas`, `Pillow`, `ImageHash`, `tqdm` 等依赖库，并确认下方第一个代码块中的 `BASE_DIR` 路径正确无误。

## 1. 全局配置与路径设定
导入所需核心依赖，并配置本地文件路径与基于关键字的动态分类映射字典。

In [1]:
import os
import cv2
import shutil
import hashlib
import random
import numpy as np
import pandas as pd
from PIL import Image
import imagehash
from tqdm import tqdm

# ==========================================
# 🛑 必须修改：请将此处的 BASE_DIR 替换为您电脑中 dataset 文件夹的实际绝对路径
# ==========================================
BASE_DIR = "./data"
GD_DIR = os.path.join(BASE_DIR, "GD")
TACO_DIR = os.path.join(BASE_DIR, "TACO")
TRASHNET_DIR = os.path.join(BASE_DIR, "TN")
OUTPUT_DIR = os.path.join(BASE_DIR, "Integrated_Dataset_384")

GD_CLASSES = ['metal', 'glass', 'biological', 'paper', 'battery', 'trash', 'cardboard', 'shoes', 'clothes', 'plastic']
TARGET_SIZE = (384, 384)
TARGET_RATIO = {'GD': 0.7, 'TACO': 0.2, 'TRASHNET': 0.1}

# 初始化最终输出目录
for cls in GD_CLASSES:
    os.makedirs(os.path.join(OUTPUT_DIR, cls), exist_ok=True)

# 动态分类映射字典 (基于关键字智能识别)
KEYWORD_RULES = {
    'glass': ['glass'], 'paper': ['paper', 'magazine', 'tissue'],
    'cardboard': ['carton', 'box', 'cardboard', 'tube'],
    'metal': ['metal', 'aluminium', 'foil', 'can', 'aerosol', 'pop tab', 'scrap'],
    'plastic': ['plastic', 'wrapper', 'bag', 'tub', 'tupperware', 'styrofoam', 'straw', 'lid', 'ring', 'container', 'cup', 'blister'],
    'biological': ['bio', 'food', 'apple', 'orange', 'banana', 'peel'],
    'battery': ['battery'], 'shoes': ['shoe'], 'clothes': ['cloth', 'garment', 'shirt', 'pants']
}
# 强制覆盖字典：处理特殊重合词语，确保分类绝对准确
OVERRIDE_MAP = {'paper cup': 'paper', 'paper bag': 'paper', 'plastified paper bag': 'paper', 'glass bottle': 'glass', 'plastic bottle cap': 'plastic'}

print("[系统] 模块 1 运行成功：依赖库已加载，全局路径与映射规则已初始化！")

[系统] 模块 1 运行成功：依赖库已加载，全局路径与映射规则已初始化！


## 2. 全局数据路径扫描器
不加载实际图像，仅高速扫描目录结构与 CSV 文件，构建包含所有目标文件路径的 `global_pool`。此步骤将统一所有数据的物理地址等待调配。

In [2]:
def scan_all_data_paths():
    print("[系统] 阶段 1：正在扫描并整合所有数据源的路径...")
    pool = {cls: {'GD': [], 'TACO': [], 'TRASHNET': []} for cls in GD_CLASSES}
    
    # [A] GD 扫描
    for cls in GD_CLASSES:
        p = os.path.join(GD_DIR, cls)
        if os.path.exists(p):
            pool[cls]['GD'] = [os.path.join(p, f) for f in os.listdir(p) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
    # [B] TrashNet 扫描
    for cls in ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']:
        p = os.path.join(TRASHNET_DIR, cls)
        if os.path.exists(p):
            pool[cls]['TRASHNET'] = [os.path.join(p, f) for f in os.listdir(p) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
    # [C] TACO 扫描 (含丢失文件容错处理)
    taco_csv = os.path.join(TACO_DIR, "meta_df.csv")
    if not os.path.exists(taco_csv):
        print(f"❌ [错误] 无法找到 TACO 的 CSV 标签文件！请检查路径: {taco_csv}")
    else:
        df_taco = pd.read_csv(taco_csv)
        unique_cats = df_taco['cat_name'].astype(str).str.lower().str.strip().unique()
        
        dynamic_map = {}
        for cat in unique_cats:
            if cat in OVERRIDE_MAP:
                dynamic_map[cat] = OVERRIDE_MAP[cat]
                continue
            mapped_class = None
            for gd_class, keywords in KEYWORD_RULES.items():
                if any(kw in cat for kw in keywords):
                    mapped_class = gd_class
                    break
            dynamic_map[cat] = mapped_class if mapped_class else 'trash'
            
        df_taco['gd_class'] = df_taco['cat_name'].astype(str).str.lower().str.strip().map(dynamic_map)
        
        missing_count = 0 
        for idx, row in df_taco.iterrows():
            # 兼容 Windows/Mac 的路径分隔符转换
            img_relative_path = str(row['img_file']).replace('/', os.sep) 
            full_img_path = os.path.join(TACO_DIR, img_relative_path)
            target_class = row['gd_class']
            
            if not os.path.exists(full_img_path):
                missing_count += 1
                if missing_count <= 3:
                    print(f"⚠️ [警告] TACO 图像丢失，已跳过: {full_img_path}")
                continue
                
            if target_class in GD_CLASSES:
                path_parts = str(row['img_file']).split('/')
                batch_name = path_parts[0] if len(path_parts) > 1 else 'batch_X'
                orig_filename = path_parts[-1].split('.')[0]
                
                bbox_data = {
                    'path': full_img_path, 'x': row['x'], 'y': row['y'], 
                    'width': row['width'], 'height': row['height'],
                    'batch': batch_name, 'orig_name': orig_filename
                }
                pool[target_class]['TACO'].append(bbox_data)
        
        if missing_count > 0:
            print(f"❌ [扫描结果] 共有 {missing_count} 张 TACO 图像文件缺失。")
        else:
            print(f"✅ [扫描结果] TACO 图像全部扫描成功！")
                
    return pool

# 提取全局路径池
global_pool = scan_all_data_paths()

[系统] 阶段 1：正在扫描并整合所有数据源的路径...
✅ [扫描结果] TACO 图像全部扫描成功！


## 3. 跨数据集防泄漏过滤 (Data Leakage Prevention)
**关键步骤**：GD 数据集在构建时可能爬取了部分 TrashNet 图像并进行了尺寸缩放。此处使用 pHash（感知哈希）计算相似度，自动将 TrashNet 中与 GD 视觉重叠的图像从 `global_pool` 中**彻底剔除**，防止模型在训练中发生数据泄露。

In [3]:
def filter_trashnet_duplicates_from_pool(pool, threshold=5):
    """
    跨维度交叉比对 GD 和 TrashNet。修改传入的 pool，主动剔除有泄露风险的重复数据。
    """
    print("\n[系统] 阶段 2：启动跨数据集防泄漏拦截机制 (pHash)...")
    total_removed = 0
    
    for cls in tqdm(GD_CLASSES, desc="跨源去重进度"):
        gd_list = pool[cls]['GD']
        tn_list = pool[cls]['TRASHNET']
        
        if not tn_list or not gd_list:
            continue
            
        # 步骤 A：提取当前类别下 GD 的视觉指纹集合
        gd_hashes = []
        for img_path in gd_list:
            try:
                img = Image.open(img_path)
                gd_hashes.append(imagehash.phash(img))
            except:
                pass
                
        # 步骤 B：对比并过滤 TrashNet 列表
        clean_tn_list = []
        for img_path in tn_list:
            try:
                img = Image.open(img_path)
                tn_h = imagehash.phash(img)
                
                # 汉明距离(Hamming Distance) <= 5 判定为同一张图
                is_duplicate = any(tn_h - gd_h <= threshold for gd_h in gd_hashes)
                
                if not is_duplicate:
                    clean_tn_list.append(img_path) # 保留安全图像
                else:
                    total_removed += 1 # 拦截克隆图像
            except:
                clean_tn_list.append(img_path)
                
        # 用清洗后的安全列表覆盖原本有风险的列表
        pool[cls]['TRASHNET'] = clean_tn_list
        
    print(f"\n[⚠️ 拦截报告] 成功拦截并从采样池中删除了 {total_removed} 张隐藏的 TrashNet 隐性重复克隆图像！")
    print("💡 结论：数据泄露(Data Leakage)风险已解除，模型泛化评估将更加真实可靠。")
    return pool

# 执行深层过滤清洗数据池
global_pool = filter_trashnet_duplicates_from_pool(global_pool, threshold=5)


[系统] 阶段 2：启动跨数据集防泄漏拦截机制 (pHash)...


跨源去重进度: 100%|████████████████████████████████████████████████████████████████████| 10/10 [00:24<00:00,  2.45s/it]


[⚠️ 拦截报告] 成功拦截并从采样池中删除了 44 张隐藏的 TrashNet 隐性重复克隆图像！
💡 结论：数据泄露(Data Leakage)风险已解除，模型泛化评估将更加真实可靠。


## 4. 图像处理与清洗工具
包含保留上下文噪声的自适应填充缩放函数（`process_and_letterbox`），以及针对集成后文件夹执行的最终收尾排重扫描（`deduplicate_integrated_dataset`）。

In [4]:
def process_and_letterbox(img_src, bbox=None, target_size=TARGET_SIZE, padding_ratio=1.5):
    """
    依据 Bounding Box 裁剪（如有），并进行 Letterbox 等比黑边填充缩放。
    改进：引入 padding_ratio，向外扩展裁剪框以保留野外环境的上下文噪声(Context Noise)。
    """
    if isinstance(img_src, str):
        img = cv2.imread(img_src)
        if img is None: 
            return None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = img_src
        
    if bbox is not None:
        h_orig, w_orig, _ = img.shape
        x = int(bbox['x'])
        y = int(bbox['y'])
        w_box = int(bbox['width'])
        h_box = int(bbox['height'])
        
        # 核心逻辑：添加上下文 Padding，保留背景特征，防止模型对边缘过度拟合
        pad_w = int(w_box * padding_ratio)
        pad_h = int(h_box * padding_ratio)
        
        new_x = max(0, x - pad_w // 2)
        new_y = max(0, y - pad_h // 2)
        new_w = min(w_orig - new_x, w_box + pad_w)
        new_h = min(h_orig - new_y, h_box + pad_h)
        
        if new_w <= 0 or new_h <= 0: 
            return None
            
        img = img[new_y:new_y+new_h, new_x:new_x+new_w]

    # Letterbox 零形变缩放
    h, w, _ = img.shape
    tw, th = target_size
    scale = min(tw / w, th / h)
    nw, nh = int(w * scale), int(h * scale)
    
    img_resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_CUBIC)
    canvas = np.zeros((th, tw, 3), dtype=np.uint8)
    dx, dy = (tw - nw) // 2, (th - nh) // 2
    canvas[dy:dy+nh, dx:dx+nw] = img_resized
    return canvas

def deduplicate_integrated_dataset(target_dir):
    """
    在集成完成后，清理各类别文件夹内部由连续拍摄或裁剪造成的重复废片。
    """
    print("\n[系统] 阶段 4：正在启动内部最终收尾清理 (MD5 + pHash)...")
    exact_removed, near_removed = 0, 0
    for cls in GD_CLASSES:
        cls_path = os.path.join(target_dir, cls)
        if not os.path.exists(cls_path): continue
            
        md5_register, phash_register = set(), {}
        for f_name in os.listdir(cls_path):
            f_path = os.path.join(cls_path, f_name)
            try:
                with open(f_path, 'rb') as fp: file_md5 = hashlib.md5(fp.read()).hexdigest()
                if file_md5 in md5_register:
                    os.remove(f_path)
                    exact_removed += 1
                    continue
                md5_register.add(file_md5)
                
                pil_img = Image.open(f_path)
                current_phash = imagehash.phash(pil_img)
                is_near_dup = False
                for registered_hash in phash_register.keys():
                    if current_phash - registered_hash <= 4:
                        os.remove(f_path)
                        near_removed += 1
                        is_near_dup = True
                        break
                if not is_near_dup: phash_register[current_phash] = f_path
            except:
                pass
    print(f"[收尾完成] 物理删除绝对内部重复: {exact_removed} 张 | 感知剔除视觉近似废片: {near_removed} 张。")

## 5. 主干合并与生成引擎 (执行核心)
利用经过严格防泄漏清洗的 `global_pool`，依照设定比例进行重采样，将处理后的图像输出至目标集成目录。

In [5]:
def run_pipeline():
    print("\n[系统] 阶段 3：执行多源深度平衡调配与文件物理生成...")
    for cls in tqdm(GD_CLASSES, desc="整体融合进度"):
        gd_list, taco_list, tn_list = global_pool[cls]['GD'], global_pool[cls]['TACO'], global_pool[cls]['TRASHNET']
        
        # 计算总配额（上限取 1200 张防止单类过载爆存）
        target_total = min(1200, len(gd_list) + len(taco_list) + len(tn_list))
        if target_total == 0: continue
        
        # 严格执行 7:2:1 比例调配
        n_gd = int(target_total * TARGET_RATIO['GD'])
        n_taco = int(target_total * TARGET_RATIO['TACO'])
        n_tn = target_total - n_gd - n_taco
        
        sampled_gd = random.sample(gd_list, min(n_gd, len(gd_list))) if gd_list else []
        sampled_taco = random.sample(taco_list, min(n_taco, len(taco_list))) if taco_list else []
        sampled_tn = random.sample(tn_list, min(n_tn, len(tn_list))) if tn_list else []
        
        # [执行] GD 主干直接 Copy
        for idx, src in enumerate(sampled_gd):
            shutil.copy(src, os.path.join(OUTPUT_DIR, cls, f"GD_{cls}_{idx}.jpg"))

        # [执行] TACO 带有上下文 Padding 裁剪及 Letterbox 缩放
        for idx, bbox_item in enumerate(sampled_taco):
            processed_box = process_and_letterbox(bbox_item['path'], bbox=bbox_item)
            if processed_box is not None:
                new_filename = f"TACO_{bbox_item['batch']}_{bbox_item['orig_name']}_crop{idx}.jpg"
                dst = os.path.join(OUTPUT_DIR, cls, new_filename)
                cv2.imwrite(dst, cv2.cvtColor(processed_box, cv2.COLOR_RGB2BGR))

        # [执行] TrashNet 全图 Letterbox 缩放
        for idx, src in enumerate(sampled_tn):
            processed_tn = process_and_letterbox(src, bbox=None)
            if processed_tn is not None:
                cv2.imwrite(os.path.join(OUTPUT_DIR, cls, f"TN_{cls}_{idx}.jpg"), cv2.cvtColor(processed_tn, cv2.COLOR_RGB2BGR))
                
    # 触发集成池的最终深度去重管道
    deduplicate_integrated_dataset(OUTPUT_DIR)
    print("\n✅ [大功告成] 恭喜！包含防泄漏架构的全流程数据集成、裁剪缩放与清洗已彻底完成！")

# 启动引擎！
if __name__ == "__main__":
    run_pipeline()


[系统] 阶段 3：执行多源深度平衡调配与文件物理生成...


整体融合进度: 100%|████████████████████████████████████████████████████████████████████| 10/10 [01:36<00:00,  9.64s/it]



[系统] 阶段 4：正在启动内部最终收尾清理 (MD5 + pHash)...
[收尾完成] 物理删除绝对内部重复: 1 张 | 感知剔除视觉近似废片: 443 张。

✅ [大功告成] 恭喜！包含防泄漏架构的全流程数据集成、裁剪缩放与清洗已彻底完成！
